# Emirates Customer Experience Intelligence
### What 800 Skytrax reviews reveal about where Emirates is winning, where it's losing, and what to fix first

---

**TL;DR** Emirates ranks #2 of 4 airlines on every single sub-rating dimension — but the gap with Qatar Airways is not uniform. It's largest in the two dimensions that most drive recommendation: value for money (−1.17 gap, highest correlation with recommendation at 0.81) and ground service (−1.27 gap, corr 0.71). Meanwhile, Emirates markets in-flight entertainment aggressively — its best sub-rating — yet entertainment has the *lowest* correlation with recommendation (0.56) of all dimensions. And when service fails entirely (lost baggage, customer service calls), the recommendation rate drops to exactly 0%.

This project applies sentiment analysis and NLP topic modeling to 800 recent Skytrax reviews across Emirates, Qatar Airways, Etihad, and Lufthansa to surface those findings in a form a CX or commercial team could actually act on.

---

## Table of contents
1. [Data collection & sources](#1)
2. [Exploratory data analysis — overall ratings](#2)
3. [Sub-rating deep dive — where the gaps are](#3)
4. [What drives recommendation? Correlation analysis](#4)
5. [Sentiment analysis (VADER)](#5)
6. [NLP topic modeling (TF-IDF + NMF)](#6)
7. [Cabin class & traveller type breakdown](#7)
8. [Key findings & business recommendations](#8)

## 1. Data collection & sources <a id='1'></a>

**Primary source: Skytrax (airlinequality.com)**
Skytrax is the most credible public airline review platform and the same organisation that issues the official airline star ratings (Emirates carries a 5-star Skytrax rating). Reviews include:
- Overall rating (1–10)
- Structured sub-ratings (1–5): seat comfort, cabin staff, food & beverages, in-flight entertainment, ground service, wifi, value for money
- Free-text review
- Verified trip flag
- Traveller type and cabin class

**Airlines scraped:** Emirates, Qatar Airways, Etihad Airways, Lufthansa (20 pages each, 10 reviews per page)

**Date range:** Reviews from 2024–2026 (most recent available at time of scraping)

**Sample size:** 800 reviews (200 per airline)

**Important caveat on bias:** Skytrax reviews skew toward aviation enthusiasts and frequent flyers — a more informed audience than general public platforms. Negative experiences are still over-represented (people are more motivated to write when dissatisfied), but less severely than on complaint-focused sites like ConsumerAffairs. All sentiment and recommendation figures in this notebook reflect the *Skytrax reviewer population*, not the general flying public.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linestyle'] = '--'

# Brand colours
COLORS = {
    'emirates':       '#D71920',
    'qatar_airways':  '#2a78d6',
    'etihad_airways': '#eda100',
    'lufthansa':      '#888780',
}
AIRLINE_LABELS = {
    'emirates':       'Emirates',
    'qatar_airways':  'Qatar Airways',
    'etihad_airways': 'Etihad Airways',
    'lufthansa':      'Lufthansa',
}
print('Libraries loaded.')

In [ ]:
df = pd.read_csv('all_airlines_analyzed.csv')
em = pd.read_csv('emirates_analyzed.csv')

print(f'Total reviews: {len(df)}')
print(f'Reviews per airline:')
print(df['airline'].value_counts())
print(f'\nColumns: {df.columns.tolist()}')
df.head(3)

## 2. Exploratory data analysis — overall ratings <a id='2'></a>

Let's start with the headline numbers: overall rating (1–10), recommendation rate, and sentiment score across all four airlines.

In [ ]:
summary = df.groupby('airline').agg(
    overall_rating=('overall_rating', 'mean'),
    rec_rate=('rec_binary', 'mean'),
    sentiment=('sentiment', 'mean'),
    count=('airline', 'size')
).sort_values('overall_rating', ascending=False).round(3)

summary.index = summary.index.map(AIRLINE_LABELS)
summary.columns = ['Overall rating /10', 'Rec rate', 'Avg sentiment', 'Reviews']
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

airlines_ordered = ['qatar_airways', 'emirates', 'lufthansa', 'etihad_airways']
labels = [AIRLINE_LABELS[a] for a in airlines_ordered]
bar_colors = [COLORS[a] for a in airlines_ordered]

# Overall rating
ratings = [df[df['airline']==a]['overall_rating'].mean() for a in airlines_ordered]
bars = axes[0].bar(labels, ratings, color=bar_colors, width=0.5, zorder=3)
axes[0].set_ylim(0, 10)
axes[0].set_title('Overall rating (out of 10)', fontsize=11, fontweight='bold', pad=10)
axes[0].set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
for bar, val in zip(bars, ratings):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'{val:.1f}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# Recommendation rate
recs = [df[df['airline']==a]['rec_binary'].mean()*100 for a in airlines_ordered]
bars2 = axes[1].bar(labels, recs, color=bar_colors, width=0.5, zorder=3)
axes[1].set_ylim(0, 100)
axes[1].set_title('Recommendation rate (%)', fontsize=11, fontweight='bold', pad=10)
axes[1].set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
for bar, val in zip(bars2, recs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.0f}%',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# Sentiment
sents = [df[df['airline']==a]['sentiment'].mean() for a in airlines_ordered]
sent_colors = ['#1baf7a' if s >= 0 else '#D71920' for s in sents]
bars3 = axes[2].bar(labels, sents, color=sent_colors, width=0.5, zorder=3)
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.5)
axes[2].set_title('Avg sentiment (VADER, −1 to +1)', fontsize=11, fontweight='bold', pad=10)
axes[2].set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
for bar, val in zip(bars3, sents):
    ypos = bar.get_height() + 0.01 if val >= 0 else bar.get_height() - 0.04
    axes[2].text(bar.get_x()+bar.get_width()/2, ypos, f'{val:+.2f}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Emirates vs competitors — headline metrics (800 Skytrax reviews, 2024–2026)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fig1_headline_metrics.png', bbox_inches='tight', dpi=150)
plt.show()

**Observation:** Qatar Airways dominates on all three metrics — 6.76/10, 67% recommendation rate, and the only airline with positive sentiment (+0.42). Emirates is a distant second at 4.06/10 and 32% rec rate, with marginally negative sentiment (−0.07). Etihad and Lufthansa are in serious trouble.

The gap between Emirates and Qatar (2.7 rating points, 35 percentage points on rec rate) is the central question this notebook investigates: *where* exactly is that gap coming from?

## 3. Sub-rating deep dive — where the gaps are <a id='3'></a>

Skytrax collects structured 1–5 star ratings on six dimensions. Let's see where Emirates loses ground vs Qatar.

In [ ]:
SUBCOLS = ['seat_comfort','cabin_staff','food_beverages',
           'inflight_entertainment','ground_service','value_for_money']
SUB_LABELS = ['Seat comfort','Cabin staff','Food & beverages',
               'Inflight entertainment','Ground service','Value for money']

sub_means = df.groupby('airline')[SUBCOLS].mean().round(2)
sub_means.index = sub_means.index.map(AIRLINE_LABELS)
sub_means.columns = SUB_LABELS
print('Sub-rating averages (1–5 scale):')
print(sub_means.to_string())

# Gap: Emirates vs Qatar
em_row = sub_means.loc['Emirates']
qa_row = sub_means.loc['Qatar Airways']
gap = (em_row - qa_row).round(2)
print('\nEmirates minus Qatar (negative = Emirates behind):')
print(gap.sort_values().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(SUB_LABELS))
w = 0.35

em_vals = sub_means.loc['Emirates'].values
qa_vals = sub_means.loc['Qatar Airways'].values

bars_em = ax.bar(x - w/2, em_vals, w, label='Emirates', color=COLORS['emirates'],
                 zorder=3, alpha=0.9)
bars_qa = ax.bar(x + w/2, qa_vals, w, label='Qatar Airways', color=COLORS['qatar_airways'],
                 zorder=3, alpha=0.9)

for bar in bars_em:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5)
for bar in bars_qa:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(SUB_LABELS, fontsize=10)
ax.set_ylim(0, 5.5)
ax.set_ylabel('Average rating (1–5)', fontsize=10)
ax.set_title('Sub-rating comparison: Emirates vs Qatar Airways\n'
             'Emirates ranks 2nd on all 6 dimensions — but the gaps differ significantly',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

# Annotate gaps
for i, (e, q) in enumerate(zip(em_vals, qa_vals)):
    ax.annotate(f'gap: {e-q:.2f}', xy=(i, min(e,q)-0.05),
                ha='center', va='top', fontsize=8, color='#444',
                style='italic')

plt.tight_layout()
plt.savefig('fig2_subrating_gap.png', bbox_inches='tight', dpi=150)
plt.show()

**Observation:** Emirates ranks #2 on every dimension — never beats Qatar anywhere. But the gaps are not uniform:
- **Ground service gap: −1.27** (Emirates 2.49 vs Qatar 3.76) — largest gap
- **Value for money gap: −1.17** (Emirates 2.36 vs Qatar 3.53) — second largest
- **Entertainment gap: −0.67** (Emirates 3.25 vs Qatar 3.92) — smallest gap

Whether those gaps matter depends on how much each dimension actually drives whether passengers recommend the airline.

## 4. What drives recommendation? Correlation analysis <a id='4'></a>

Not all sub-ratings matter equally. Let's compute the Pearson correlation between each sub-rating and the binary "recommended" outcome across all 800 reviews.

In [ ]:
corr = df[SUBCOLS + ['rec_binary']].corr()['rec_binary'].drop('rec_binary').sort_values(ascending=False)
corr.index = SUB_LABELS
print('Correlation with recommendation rate (higher = stronger predictor):')
print(corr.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bar_cols = ['#D71920' if i == 0 else '#1baf7a' if i == len(corr)-1 else '#2a78d6'
            for i in range(len(corr))]
bars = ax.barh(corr.index[::-1], corr.values[::-1], color=bar_cols[::-1],
               height=0.5, zorder=3)

for bar, val in zip(bars, corr.values[::-1]):
    ax.text(val + 0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0.4, 0.88)
ax.set_xlabel('Pearson correlation with "recommended"', fontsize=10)
ax.set_title('Which sub-rating most predicts whether a passenger recommends the airline?\n'
             'Value for money (0.81) and ground service (0.71) dominate — entertainment is weakest (0.56)',
             fontsize=11, fontweight='bold')

red_p = mpatches.Patch(color='#D71920', label='Strongest predictor')
blue_p = mpatches.Patch(color='#2a78d6', label='Mid-range')
green_p = mpatches.Patch(color='#1baf7a', label='Weakest predictor')
ax.legend(handles=[red_p, blue_p, green_p], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('fig3_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

**The core finding of this analysis:**

| Dimension | Emirates score | Qatar score | Gap | Correlation with rec |
|---|---|---|---|---|
| Value for money | 2.36 | 3.53 | **−1.17** | **0.81** (strongest) |
| Ground service | 2.49 | 3.76 | **−1.27** | **0.71** |
| Inflight entertainment | 3.25 | 3.92 | −0.67 | **0.56** (weakest) |

Emirates' largest competitive gaps are in the dimensions that most drive recommendation. Its smallest gap is in the dimension that least drives recommendation — yet entertainment is what Emirates most heavily markets (its ICE system). This is a meaningful strategic misalignment.

## 5. Sentiment analysis (VADER) <a id='5'></a>

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon-based sentiment model tuned for social reviews. It outputs a compound score from −1 (most negative) to +1 (most positive).

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
df['sentiment'] = df['review_text'].apply(
    lambda t: analyzer.polarity_scores(str(t))['compound']
)

print('Sentiment distribution by airline:')
print(df.groupby('airline')['sentiment'].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)

for ax, airline in zip(axes, ['qatar_airways','emirates','lufthansa','etihad_airways']):
    data = df[df['airline']==airline]['sentiment']
    color = COLORS[airline]
    ax.hist(data, bins=20, color=color, alpha=0.8, edgecolor='white', linewidth=0.5, zorder=3)
    ax.axvline(data.mean(), color='black', linewidth=1.5, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle=':')
    ax.set_title(AIRLINE_LABELS[airline], fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Sentiment score', fontsize=9)
    ax.text(0.05, 0.95, f'Mean: {data.mean():+.2f}', transform=ax.transAxes,
            fontsize=9, va='top', fontweight='bold')

plt.suptitle('Sentiment score distribution by airline (VADER compound score)',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_sentiment_dist.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. NLP topic modeling (TF-IDF + NMF) <a id='6'></a>

We apply TF-IDF vectorization and Non-negative Matrix Factorization (NMF) to extract latent topics from Emirates' 200 reviews. NMF works well on short review texts and produces more interpretable topics than LDA for this kind of data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

def clean(text):
    text = str(text).lower()
    text = re.sub(r'✅ trip verified \|', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text

em['clean_text'] = em['review_text'].apply(clean)

STOPWORDS = ['the','a','an','and','to','of','in','was','is','i','my','we',
             'for','on','at','with','had','were','our','they','this','that',
             'it','not','but','from','have','been','trip','verified','flight',
             'emirates','would','when','as','me','be','so','no','are','by',
             'very','just','even','will','also','than','there','then','them',
             'their','after','all','more','about','which','one','time','got']

tfidf = TfidfVectorizer(max_df=0.85, min_df=3, stop_words=STOPWORDS, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(em['clean_text'])

nmf = NMF(n_components=6, random_state=42)
nmf.fit(tfidf_matrix)

feature_names = tfidf.get_feature_names_out()
print('Top 6 topics in Emirates reviews:\n')
for i, topic in enumerate(nmf.components_):
    top_words = [feature_names[j] for j in topic.argsort()[:-12:-1]]
    print(f'Topic {i}: {" | ".join(top_words)}')

In [ ]:
em['sentiment'] = em['review_text'].apply(
    lambda t: analyzer.polarity_scores(str(t))['compound']
)
em['rec_binary'] = (em['recommended'] == 'yes').astype(int)

topic_assignments = nmf.transform(tfidf_matrix).argmax(axis=1)
em['topic'] = topic_assignments

TOPIC_NAMES = {
    0: 'Cabin class comparisons',
    1: 'Positive in-flight experience',
    2: 'Crew & onboard service',
    3: 'Dubai airport & delays',
    4: 'Baggage & customer service',
    5: 'Staff behavior & assistance'
}

topic_summary = em.groupby('topic').agg(
    count=('topic','size'),
    sentiment=('sentiment','mean'),
    rec_rate=('rec_binary','mean')
).round(3)
topic_summary['label'] = topic_summary.index.map(TOPIC_NAMES)
topic_summary = topic_summary.sort_values('sentiment', ascending=False)
print(topic_summary[['label','count','sentiment','rec_rate']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sorted_topics = topic_summary.sort_values('sentiment')
labels = [TOPIC_NAMES[i] for i in sorted_topics.index]
sent_vals = sorted_topics['sentiment'].values
rec_vals = sorted_topics['rec_rate'].values * 100

sent_cols = ['#1baf7a' if v > 0.2 else '#eda100' if v > -0.2 else '#D71920' for v in sent_vals]
rec_cols = ['#1baf7a' if v > 50 else '#eda100' if v > 20 else '#D71920' for v in rec_vals]

axes[0].barh(labels, sent_vals, color=sent_cols, height=0.55, zorder=3)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Avg sentiment score', fontsize=10)
axes[0].set_title('Sentiment by topic', fontsize=11, fontweight='bold')
for i, val in enumerate(sent_vals):
    axes[0].text(val + 0.01 if val >= 0 else val - 0.01, i,
                 f'{val:+.2f}', va='center', ha='left' if val >= 0 else 'right',
                 fontsize=9, fontweight='bold')

axes[1].barh(labels, rec_vals, color=rec_cols, height=0.55, zorder=3)
axes[1].set_xlabel('Recommendation rate (%)', fontsize=10)
axes[1].set_title('Recommendation rate by topic', fontsize=11, fontweight='bold')
axes[1].set_xlim(0, 100)
for i, val in enumerate(rec_vals):
    axes[1].text(val + 0.5, i, f'{val:.0f}%', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Emirates review topics — NMF topic model (6 topics, 200 reviews)',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_topics.png', bbox_inches='tight', dpi=150)
plt.show()

**Topic model findings:**

| Topic | Sentiment | Rec rate | Key implication |
|---|---|---|---|
| Positive in-flight experience | +0.69 | **79%** | When the product works, it works well |
| Crew & onboard service | +0.36 | **71%** | Crew quality is a real strength |
| Cabin class comparisons | +0.14 | 46% | Mixed — Economy pulls this down |
| Staff behavior & assistance | −0.19 | 19% | Inconsistent staff a problem |
| Dubai airport & delays | −0.33 | 8% | Ground ops are a critical failure point |
| **Baggage & customer service** | **−0.57** | **0%** | **Not a single recommendation** |

The 0% recommendation rate for baggage & customer service is the most actionable single number in this analysis. When Emirates fails at service recovery, it loses passengers permanently.

## 7. Cabin class & traveller type breakdown <a id='7'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cabin class
cabin_sent = em.groupby('seat_type')['sentiment'].mean().sort_values(ascending=False)
cabin_cols = ['#1baf7a' if v >= 0 else '#D71920' for v in cabin_sent.values]
axes[0].barh(cabin_sent.index[::-1], cabin_sent.values[::-1],
             color=cabin_cols[::-1], height=0.45, zorder=3)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Avg sentiment', fontsize=10)
axes[0].set_title('Sentiment by cabin class (Emirates)', fontsize=11, fontweight='bold')
for i, (idx, val) in enumerate(zip(cabin_sent.index[::-1], cabin_sent.values[::-1])):
    axes[0].text(val + 0.01 if val >= 0 else val - 0.01, i, f'{val:+.2f}',
                 va='center', ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

# Traveller type
trav_sent = em.groupby('traveller_type')['sentiment'].mean().sort_values(ascending=False)
trav_cols = ['#1baf7a' if v >= 0 else '#D71920' for v in trav_sent.values]
axes[1].barh(trav_sent.index[::-1], trav_sent.values[::-1],
             color=trav_cols[::-1], height=0.45, zorder=3)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Avg sentiment', fontsize=10)
axes[1].set_title('Sentiment by traveller type (Emirates)', fontsize=11, fontweight='bold')
for i, (idx, val) in enumerate(zip(trav_sent.index[::-1], trav_sent.values[::-1])):
    axes[1].text(val + 0.01 if val >= 0 else val - 0.01, i, f'{val:+.2f}',
                 va='center', ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

plt.suptitle('Emirates — sentiment breakdown by cabin class and traveller type',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_cabin_breakdown.png', bbox_inches='tight', dpi=150)
plt.show()

**Two-tier experience finding:**
- Premium Economy (+0.60) and First Class (+0.35) passengers are net positive
- Economy class passengers are net negative (−0.23)
- Business travellers — the highest lifetime value segment — are net negative (−0.20)
- Only solo leisure travellers are marginally positive (+0.07)

Emirates' premium product works. Its mass product and business product do not.

## 8. Key findings & business recommendations <a id='8'></a>

---

### Finding 1 — Highest priority
**Baggage & customer service failures produce a 0% recommendation rate**

The 35 reviews classified under the "baggage & customer service" topic contain zero recommendations. The sentiment score of −0.57 is the most negative of any topic. Key phrases in these reviews: *lost luggage, customer service, months waiting, no response, refund refused.* When Emirates fails at service recovery, it loses the passenger permanently — and likely their network, given that dissatisfied passengers are far more likely to share their experience than satisfied ones.

**Recommended action:** Service recovery protocols and response time SLAs for baggage claims and refund requests should be the first CX investment, before any product enhancement.

---

### Finding 2 — Strategic
**Emirates underperforms most where it matters most**

Value for money (Emirates 2.36 vs Qatar 3.53, gap −1.17) is the strongest predictor of recommendation (corr 0.81). Ground service (Emirates 2.49 vs Qatar 3.76, gap −1.27) is the second strongest (corr 0.71). These are not coincidentally the two areas where Emirates' competitive gap is also largest.

In-flight entertainment — what Emirates markets most heavily — has the *smallest* correlation with recommendation (0.56) and only the fifth-largest gap. This suggests a marketing and investment strategy that is misaligned with what actually drives passenger loyalty.

---

### Finding 3 — Product
**A two-tier experience problem — premium passengers are satisfied, economy are not**

First Class sentiment (+0.35) and Premium Economy (+0.60) are both positive. Economy class (−0.23) is negative. Since Economy class carries the vast majority of Emirates' passengers, this creates a structural brand-risk problem: the typical Emirates passenger leaves with a net negative experience, even as the brand's marketing implies a premium product for all.

---

### Finding 4 — Marketing
**Emirates is #1 at what matters least**

Emirates' highest-scoring sub-rating is inflight entertainment (3.25/5 — the best relative score vs the field, where Qatar scores 3.92). Yet entertainment has the lowest correlation with whether a passenger recommends the airline (0.56). Emirates may be winning the ICE war while losing the loyalty war.

---

## Methodology notes

- **Scraping:** Public Skytrax reviews fetched with `requests` + `BeautifulSoup`. Script included in the repository. Respects a polite crawl delay between requests.
- **Sentiment:** VADER (compound score). Chosen for speed and interpretability on review text. For production use, a fine-tuned transformer (e.g. `cardiffnlp/twitter-roberta-base-sentiment`) would improve accuracy, particularly on mixed-sentiment reviews.
- **Topic model:** TF-IDF vectorization (1–2 grams, max_df 0.85, min_df 3) + NMF (6 components). Topic count chosen by coherence inspection; 6 produced the most interpretable topic separation on this corpus.
- **Limitations:** 200 reviews per airline is sufficient for directional findings but not for statistically robust sub-group analysis (e.g. sentiment by route). Scaling the scraper to 500+ reviews per airline and adding route-level filtering would substantially strengthen the analysis.

---

*If this analysis was useful, please upvote and feel free to fork and extend it. Feedback and pull requests welcome on [GitHub](https://github.com).*